# Task 01 — Bronze: Raw Orders Load

**Workshop: Final Pipeline | Layer 1 of 3**

## Goal

Load `orders_batch.json` from the Volume into a Bronze Delta table.
Bronze = raw data + metadata only. No business logic.

```
Volume: DATASET_PATH/orders/orders_batch.json
            |
            v  spark.read + explicit schema
            |  + _load_ts (timestamp)  + _source_file (path)
            v
    bronze.lab_orders
```

## Expected schema

| Column | Type | Notes |
|--------|------|-------|
| order_id | StringType | Not null |
| customer_id | StringType | |
| product_id | StringType | |
| store_id | StringType | |
| order_datetime | StringType | Kept as String — Silver casts it |
| quantity | StringType | |
| unit_price | StringType | |
| discount_percent | StringType | |
| total_amount | StringType | |
| payment_method | StringType | |
| status | StringType | |
| _load_ts | TimestampType | Added: load time |
| _source_file | StringType | Added: source file path |

> Bronze rule: do not transform, calculate, or filter — that is Silver's job.


In [0]:
%run ../../../setup/00_setup

In [0]:
dbutils.widgets.text("source_path", DATASET_PATH, "Source Path")
dbutils.widgets.text("catalog",     CATALOG,       "Catalog")
dbutils.widgets.text("schema",      BRONZE_SCHEMA, "Bronze Schema")

source_path  = dbutils.widgets.get("source_path")
catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")

orders_path  = f"{source_path}/orders/orders_batch.json"
target_table = f"{catalog}.{schema}.lab_orders"

print(f"Source : {orders_path}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import `StructType`, `StructField`, and `StringType` — all 11 JSON columns are stored as `StringType` at the Bronze layer.
Also import `current_timestamp` and `col` from `pyspark.sql.functions` (we'll use `col` to access `_metadata.file_path`).

**💡 Guidance — Exercise 1**

Import the types and functions needed to define the JSON schema and add Bronze metadata columns.

**Two import groups**
- From `pyspark.sql.types`: the schema building blocks — `StructType`, `StructField`, and `StringType`. All 11 JSON columns are kept as `StringType` at the Bronze layer — no numeric casting here.
- From `pyspark.sql.functions`: two metadata utilities — `current_timestamp()` returns the current server time as a column expression, and `col()` which we'll use to access `_metadata.file_path` (the Unity Catalog way to get source file path). Both are used inside `.withColumn()` after the file is loaded.

**Things to think about**
- Why is `_metadata.file_path` more useful than hardcoding the path with `lit("...")`?
- What does `current_timestamp()` return if called outside a `spark.read` context?

In [0]:
# TODO: Import StructType, StructField, StringType from pyspark.sql.types
# TODO: Import current_timestamp and col from pyspark.sql.functions

# Your code here:


---

## Exercise 2 — Define the schema

Define a `StructType` for the **11 JSON columns** listed in the table above.
`order_id` must have `nullable=False`; all others `nullable=True`.


**💡 Guidance — Exercise 2**

Define an explicit `StructType` schema for the 11 JSON columns.

**Constructing the schema**
Create a `StructType` containing one `StructField` per column. Each field takes three arguments: the column name (string), a type instance, and a `nullable` boolean. For this dataset: `order_id` must have `nullable=False`; all other columns `nullable=True`.

**All 11 columns use `StringType()`**
Bronze is a 1-to-1 copy of the source — no type coercion, no interpretation. Numeric fields (`quantity`, `unit_price`, `discount_percent`, `total_amount`) are stored as raw strings, exactly as they appear in the JSON. Casting to numeric types is Silver's responsibility.

**Things to think about**
- What happens at read time if a field contains a non-numeric value and you have defined it as `IntegerType`?
- Why is `nullable=False` on `order_id` important for downstream joins and quality checks?


In [0]:
# TODO: Define orders_schema as StructType with 11 fields
# All columns → StringType()
# order_id → nullable=False, all others → nullable=True

# Expected columns:
# order_id, customer_id, product_id, store_id, order_datetime,
# quantity, unit_price, discount_percent, total_amount, payment_method, status

orders_schema = StructType([
    # Your code here:
    
])

orders_schema  # display schema

---

## Exercise 3 — Read the JSON file and add metadata columns

Read `orders_path` using your schema.
Add `_load_ts` and `_source_file` with `.withColumn()`.
Name the result `orders_raw`.


**💡 Guidance — Exercise 3**

Read the JSON file using your schema and immediately attach two Bronze metadata columns.

**Loading the file**
Use the standard reader chain: `spark.read.format("json").schema(orders_schema).load(orders_path)`. Always pass an explicit schema — never use `inferSchema` in production pipelines because it requires a full double-scan of the data.

**Metadata columns to add**
Chain two `.withColumn()` calls on the loaded DataFrame:
- `_load_ts` → `current_timestamp()` — records the exact server timestamp when the batch was ingested
- `_source_file` → `col("_metadata.file_path")` — records the full path of the source file being read (Unity Catalog method)

Using `_metadata.file_path` instead of `lit(orders_path)` means the value resolves correctly even when the same notebook is used with different source files. Store the final DataFrame as `orders_raw`.

**Things to think about**
- What does the Bronze layer rule say about filtering or transforming the data at this stage?
- Would `lit(orders_path)` give different results from `_metadata.file_path` in a multi-file wildcard load?

In [0]:
# TODO: Read orders_path as JSON using orders_schema
# TODO: Add _load_ts column with current_timestamp()
# TODO: Add _source_file column with col("_metadata.file_path")

# Step 1: spark.read.format(...).schema(...).load(...)
# Step 2: .withColumn("_load_ts", ...)
# Step 3: .withColumn("_source_file", ...)

orders_raw = None  # Replace with your code

In [0]:
print(f"Rows loaded : {orders_raw.count():,}")
orders_raw.printSchema()
display(orders_raw.limit(5))

---

## Exercise 4 — Write to Bronze as a managed Delta table


**💡 Guidance — Exercise 4**

Write `orders_raw` to a managed Delta table as the Bronze layer of the pipeline.

**Write options**
- `format("delta")` — store as a Delta table (ACID, version history, time travel)
- `mode("overwrite")` — fully replace the table on each pipeline run (full refresh)
- `option("overwriteSchema", "true")` — allow the table schema to change on re-runs (e.g., a new column added to the source JSON)
- `saveAsTable(target_table)` — register as a **managed** Unity Catalog table; Databricks controls both the metadata and the physical Parquet files

**Full refresh vs. append**
In this pipeline Bronze is always overwritten, not appended. The same raw file is always re-processed to ensure idempotency — running the notebook twice produces the same result.

**Things to think about**
- What would happen if you used `mode("append")` on every pipeline run?
- After writing, the notebook prints `DESCRIBE HISTORY` output — what operation name do you expect to see for the write?

In [0]:
# TODO: Write orders_raw to target_table as a managed Delta table
# Required options:
#   - format: "delta"
#   - mode: "overwrite"
#   - option("overwriteSchema", "true")
#   - saveAsTable(target_table)

# Your code here:


In [0]:
import json

row_count = spark.read.table(target_table).count()
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))